# 06 — Deliverables

Generates the Excel workbooks from pipeline results.
The slide deck (`slides.md`) is edited directly.

1. **Papers Excel** (`synbio_papers.xlsx`) — cluster summary, paper assignments, precedence
2. **Teams Excel** (`igem_teams.xlsx`) — cluster summary, team assignments, precedence

Run after all upstream notebooks (00–05) have been executed.

In [23]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent
ASSETS_DIR = ROOT_DIR / "assets"
REPORTS_DIR = ASSETS_DIR / "reports"
OUT_DIR = Path.cwd()

# Input paths — reports
CLUSTER_SUMMARY_PAPERS = REPORTS_DIR / "cluster_summary_papers.tsv"
CLUSTER_SUMMARY_IGEM = REPORTS_DIR / "cluster_summary_igem.tsv"
PAPERS_HIERARCHY_MAP = REPORTS_DIR / "papers_topic_hierarchy_map.tsv"
PAPERS_HIERARCHY_NAMES = REPORTS_DIR / "papers_topic_name_hierarchy.tsv"
TEAMS_HIERARCHY_MAP = REPORTS_DIR / "teams_topic_hierarchy_map.tsv"
TEAMS_HIERARCHY_NAMES = REPORTS_DIR / "teams_topic_name_hierarchy.tsv"
LIT_PRECEDED = REPORTS_DIR / "literature_preceded.tsv"
IGEM_PRECEDED = REPORTS_DIR / "igem_preceded.tsv"

# Input paths — source data
OPENALEX_PATH = ASSETS_DIR / "synbio_openalex.txt"
IGEM_PATH = ASSETS_DIR / "igem.txt"
PAPERS_TOPIC_NAMES = ASSETS_DIR / "topic_models" / "papers_topic_names.txt"
TEAMS_TOPIC_NAMES = ASSETS_DIR / "topic_models" / "teams_topic_names.txt"

# Output paths
OUT_PAPERS_XLSX = OUT_DIR / "synbio_papers.xlsx"
OUT_TEAMS_XLSX = OUT_DIR / "igem_teams.xlsx"

print("Paths configured ✓")

Paths configured ✓


In [25]:
# Load all datasets
print("Loading datasets …")

cluster_papers = pd.read_csv(CLUSTER_SUMMARY_PAPERS, sep="\t")
cluster_igem = pd.read_csv(CLUSTER_SUMMARY_IGEM, sep="\t")

papers_hier_map = pd.read_csv(PAPERS_HIERARCHY_MAP, sep="\t")
papers_hier_names = pd.read_csv(PAPERS_HIERARCHY_NAMES, sep="\t")

teams_hier_map = pd.read_csv(TEAMS_HIERARCHY_MAP, sep="\t")
teams_hier_names = pd.read_csv(TEAMS_HIERARCHY_NAMES, sep="\t")

lit_preceded = pd.read_csv(LIT_PRECEDED, sep="\t")
igem_preceded = pd.read_csv(IGEM_PRECEDED, sep="\t")

openalex = pd.read_csv(OPENALEX_PATH, sep="\t")
igem = pd.read_csv(IGEM_PATH, sep="\t")
papers_topic_names = pd.read_csv(PAPERS_TOPIC_NAMES, sep="\t")
teams_topic_names = pd.read_csv(TEAMS_TOPIC_NAMES, sep="\t")

datasets = {
    "cluster_papers": cluster_papers,
    "cluster_igem": cluster_igem,
    "papers_hier_map": papers_hier_map,
    "papers_hier_names": papers_hier_names,
    "teams_hier_map": teams_hier_map,
    "teams_hier_names": teams_hier_names,
    "lit_preceded": lit_preceded,
    "igem_preceded": igem_preceded,
    "openalex": openalex,
    "igem": igem,
}
for name, df in datasets.items():
    print(f"  {name:25s}: {len(df):>6,} rows × {len(df.columns)} cols")
print("Done ✓")

Loading datasets …
  cluster_papers           :    263 rows × 10 cols
  cluster_igem             :    161 rows × 12 cols
  papers_hier_map          : 24,202 rows × 4 cols
  papers_hier_names        :    263 rows × 6 cols
  teams_hier_map           :  4,548 rows × 4 cols
  teams_hier_names         :    161 rows × 6 cols
  lit_preceded             :     36 rows × 7 cols
  igem_preceded            :     34 rows × 7 cols
  openalex                 : 24,202 rows × 15 cols
  igem                     :  4,548 rows × 27 cols
Done ✓


In [26]:
# Papers Excel — prepare sheets
print("Preparing papers Excel …")

# Helper: map low → hierarchy names
p_name_cols = papers_hier_names[["low", "mid", "high", "mid_name", "high_name", "global_name"]].copy()
p_name_cols = p_name_cols.rename(columns={
    "high_name": "high_level_cluster_name",
    "mid_name": "mid_level_cluster_name",
    "global_name": "low_level_cluster_name",
})

# Sheet 1: Cluster Summary with hierarchy names prepended
sheet1_papers = cluster_papers.rename(columns={"topic": "low"}).merge(
    p_name_cols[["low", "mid", "high", "mid_level_cluster_name", "high_level_cluster_name", "low_level_cluster_name"]],
    on="low", how="left"
).merge(
    papers_topic_names[["topic", "description"]].rename(columns={"topic": "low"}),
    on="low", how="left"
)
existing_cols = [c for c in cluster_papers.columns if c not in ("topic", "global_name")]
sheet1_papers = sheet1_papers[
    ["high_level_cluster_name", "high", "mid_level_cluster_name", "mid", "low_level_cluster_name", "low"] + existing_cols + ["description"]
].sort_values(["high", "mid", "low"])

# Sheet 2: Paper assignments with metadata (exclude outlier topic -1)
sheet2_papers = (
    papers_hier_map
    .query("low != -1")
    .merge(p_name_cols[["low", "mid_level_cluster_name", "high_level_cluster_name", "low_level_cluster_name"]], on="low", how="left")
    .merge(
        openalex[["id", "doi", "title", "cited_by_count", "publication_year", "source_name"]],
        left_on="ID", right_on="id", how="left"
    )
    .drop(columns=["id"])
)
sheet2_papers = sheet2_papers[
    ["high", "high_level_cluster_name", "mid", "mid_level_cluster_name", "low", "low_level_cluster_name",
     "ID", "doi", "title", "cited_by_count", "publication_year", "source_name"]
].sort_values(["high", "mid", "low"])


# Sheet 3: Literature preceded
sheet3_papers = lit_preceded.rename(columns={"topic": "low", "global_name": "low_level_cluster_name"}).copy()


print(f"  Sheet 1 (Cluster Summary) : {len(sheet1_papers):,} rows")
print(f"  Sheet 2 (Paper Assignments): {len(sheet2_papers):,} rows")
print(f"  Sheet 3 (Lit Preceded)     : {len(sheet3_papers):,} rows")

Preparing papers Excel …
  Sheet 1 (Cluster Summary) : 263 rows
  Sheet 2 (Paper Assignments): 15,185 rows
  Sheet 3 (Lit Preceded)     : 36 rows


In [27]:
# Write papers Excel
from openpyxl.utils import get_column_letter

def _autofit_columns(writer, sheet_name, df):
    """Set column widths to max of header/content length (capped at 50)."""
    ws = writer.sheets[sheet_name]
    for i, col in enumerate(df.columns, start=1):
        max_len = max(
            df[col].astype(str).map(len).max(),
            len(str(col))
        )
        ws.column_dimensions[get_column_letter(i)].width = min(max_len + 2, 50)

with pd.ExcelWriter(OUT_PAPERS_XLSX, engine="openpyxl") as writer:
    sheet1_papers.to_excel(writer, sheet_name="Cluster Summary", index=False)
    sheet2_papers.to_excel(writer, sheet_name="Paper Assignments", index=False)
    sheet3_papers.to_excel(writer, sheet_name="Literature Preceded", index=False)
    for sn, df in [("Cluster Summary", sheet1_papers),
                    ("Paper Assignments", sheet2_papers),
                    ("Literature Preceded", sheet3_papers)]:
        _autofit_columns(writer, sn, df)

print(f"Saved → {OUT_PAPERS_XLSX.name} ({OUT_PAPERS_XLSX.stat().st_size / 1024:.0f} KB)")

Saved → synbio_papers.xlsx (1893 KB)


In [28]:
# Teams Excel — prepare sheets
print("Preparing teams Excel …")

t_name_cols = teams_hier_names[["low", "mid", "high", "mid_name", "high_name", "global_name"]].copy()
t_name_cols = t_name_cols.rename(columns={
    "high_name": "high_level_cluster_name",
    "mid_name": "mid_level_cluster_name",
    "global_name": "low_level_cluster_name",
})

# Sheet 1: Cluster Summary with hierarchy names
sheet1_teams = cluster_igem.rename(columns={"topic": "low"}).merge(
    t_name_cols[["low", "mid", "high", "mid_level_cluster_name", "high_level_cluster_name", "low_level_cluster_name"]],
    on="low", how="left"
).merge(
    teams_topic_names[["topic", "description"]].rename(columns={"topic": "low"}),
    on="low", how="left"
)
existing_cols_t = [c for c in cluster_igem.columns if c not in ("topic", "global_name")]
sheet1_teams = sheet1_teams[
    ["high_level_cluster_name", "high", "mid_level_cluster_name", "mid", "low_level_cluster_name", "low"] + existing_cols_t + ["description"]
].sort_values(["high", "mid", "low"])

# Sheet 2: Team assignments with metadata (exclude outlier topic -1)
sheet2_teams = (
    teams_hier_map
    .query("low != -1")
    .merge(t_name_cols[["low", "mid_level_cluster_name", "high_level_cluster_name", "low_level_cluster_name"]], on="low", how="left")
    .merge(
        igem[["UT", "TI", "Institutions", "Year_y"]],
        on="UT", how="left"
    )
    .rename(columns={"TI": "Title", "Year_y": "Year"})
)
sheet2_teams = sheet2_teams[
    ["high", "high_level_cluster_name", "mid", "mid_level_cluster_name", "low", "low_level_cluster_name",
     "UT", "Title", "Institutions", "Year"]
].sort_values(["high", "mid", "low"])

# Sheet 3: iGEM preceded
sheet3_teams = igem_preceded.rename(columns={"topic": "low", "global_name": "low_level_cluster_name"}).copy()

print(f"  Sheet 1 (Cluster Summary) : {len(sheet1_teams):,} rows")

print(f"  Sheet 2 (Team Assignments): {len(sheet2_teams):,} rows")
print(f"  Sheet 3 (iGEM Preceded)   : {len(sheet3_teams):,} rows")

Preparing teams Excel …
  Sheet 1 (Cluster Summary) : 161 rows
  Sheet 2 (Team Assignments): 4,548 rows
  Sheet 3 (iGEM Preceded)   : 34 rows


In [29]:
# Write teams Excel
with pd.ExcelWriter(OUT_TEAMS_XLSX, engine="openpyxl") as writer:
    sheet1_teams.to_excel(writer, sheet_name="Cluster Summary", index=False)
    sheet2_teams.to_excel(writer, sheet_name="Team Assignments", index=False)
    sheet3_teams.to_excel(writer, sheet_name="iGEM Preceded", index=False)
    for sn, df in [("Cluster Summary", sheet1_teams),
                    ("Team Assignments", sheet2_teams),
                    ("iGEM Preceded", sheet3_teams)]:
        _autofit_columns(writer, sn, df)

print(f"Saved → {OUT_TEAMS_XLSX.name} ({OUT_TEAMS_XLSX.stat().st_size / 1024:.0f} KB)")

Saved → igem_teams.xlsx (537 KB)


In [30]:
# Summary of outputs
outputs = [OUT_PAPERS_XLSX, OUT_TEAMS_XLSX]
print("=" * 60)
print("Deliverables generated")
print("=" * 60)
for p in outputs:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:30s}  {size_kb:>8.1f} KB")
print()
print(f"synbio_papers.xlsx:")
print(f"  Cluster Summary     : {len(sheet1_papers):>6,} rows")
print(f"  Paper Assignments   : {len(sheet2_papers):>6,} rows")
print(f"  Literature Preceded : {len(sheet3_papers):>6,} rows")
print()
print(f"igem_teams.xlsx:")
print(f"  Cluster Summary     : {len(sheet1_teams):>6,} rows")
print(f"  Team Assignments    : {len(sheet2_teams):>6,} rows")
print(f"  iGEM Preceded       : {len(sheet3_teams):>6,} rows")

Deliverables generated
  synbio_papers.xlsx                1892.7 KB
  igem_teams.xlsx                    536.6 KB

synbio_papers.xlsx:
  Cluster Summary     :    263 rows
  Paper Assignments   : 15,185 rows
  Literature Preceded :     36 rows

igem_teams.xlsx:
  Cluster Summary     :    161 rows
  Team Assignments    :  4,548 rows
  iGEM Preceded       :     34 rows
